<a href="https://colab.research.google.com/github/DanielCSIm/Mastering-Python-Scripting-for-System-Administrators-/blob/master/classPrepPyspark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import urllib.request
url="https://github.com/DataTalksClub/nyc-tlc-data/releases/download/yellow/yellow_tripdata_2019-01.csv.gz"
file_path="yellow_tripdata_2019-01.csv.gz"

urllib.request.urlretrieve(url,file_path)
print("Dataset downloaded!")

Dataset downloaded!


In [2]:
# To compare Pandas and Apache Spark
import pandas as pd
import time

# To load dataset
start_time=time.time()
df=pd.read_csv("yellow_tripdata_2019-01.csv.gz",compression='gzip')
load_time=time.time()-start_time
print(f"Pandas: Data loaded in {load_time:.2f} seconds")

Pandas: Data loaded in 29.40 seconds


In [3]:
df.head(40)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge
0,1,2019-01-01 00:46:40,2019-01-01 00:53:20,1,1.50,1,N,151,239,1,7.0,0.5,0.5,1.65,0.00,0.3,9.95,NaN
1,1,2019-01-01 00:59:47,2019-01-01 01:18:59,1,2.60,1,N,239,246,1,14.0,0.5,0.5,1.00,0.00,0.3,16.30,NaN
2,2,2018-12-21 13:48:30,2018-12-21 13:52:40,3,0.00,1,N,236,236,1,4.5,0.5,0.5,0.00,0.00,0.3,5.80,NaN
3,2,2018-11-28 15:52:25,2018-11-28 15:55:45,5,0.00,1,N,193,193,2,3.5,0.5,0.5,0.00,0.00,0.3,7.55,NaN
4,2,2018-11-28 15:56:57,2018-11-28 15:58:33,5,0.00,2,N,193,193,2,52.0,0.0,0.5,0.00,0.00,0.3,55.55,NaN
5,2,2018-11-28 16:25:49,2018-11-28 16:28:26,5,0.00,1,N,193,193,2,3.5,0.5,0.5,0.00,5.76,0.3,13.31,NaN
6,2,2018-11-28 16:29:37,2018-11-28 16:33:43,5,0.00,2,N,193,193,2,52.0,0.0,0.5,0.00,0.00,0.3,55.55,NaN
7,1,2019-01-01 00:21:28,2019-01-01 00:28:37,1,1.30,1,N,163,229,1,6.5,0.5,0.5,1.25,0.00,0.3,9.05,NaN
8,1,2019-01-01 00:32:01,2019-01-01 00:45:39,1,3.70,1,N,229,7,1,13.5,0.5,0.5,3.70,0.00,0.3,18.50,NaN
9,1,2019-01-01 00:57:32,2019-01-01 01:09:32,2,2.10,1,N,141,234,1,10.0,0.5,0.5,1.70,0.00,0.3,13.00,NaN


In [4]:
# To filter trips longer than 10 miles
start_time=time.time()
long_trips=df[df['trip_distance']>10]
filter_time=time.time()-start_time
print(f"Pandas: Filter operations took {filter_time:.2f} seconds")
print(len(long_trips)/len(df))

Pandas: Filter operations took 0.20 seconds
0.055259584506204655


In [5]:
# To calculate average trip distance
start_time=time.time()
avg_distance=df['trip_distance'].mean()
agg_time=time.time()-start_time
print(f"Pandas: Aggregation took {agg_time:.2f} seconds")
print(f"Pandas: Average trip distance is {avg_distance:.2f} miles")

Pandas: Aggregation took 0.03 seconds
Pandas: Average trip distance is 2.80 miles


In [6]:
# Spark
from pyspark.sql import SparkSession
from pyspark.sql.functions import mean
import time

# To initialize SparkSession
spark=SparkSession.builder.appName("NYC_Taxi_Comparison").getOrCreate()

# To load dataset
start_time=time.time()
df=spark.read.csv("yellow_tripdata_2019-01.csv.gz",header=True,inferSchema=True)
load_time=time.time()-start_time
print(f"Spark: Data loaded in {load_time:.2f} seconds")

Spark: Data loaded in 54.55 seconds


In [7]:
# To filter trips longer than 10 miles
start_time=time.time()
long_trips=df.filter(df['trip_distance']>10)
long_trips_count=long_trips.count() # why do I need this?
filter_time=time.time()-start_time
print(f"Spark: Filter operation took {filter_time:.2f} seconds")

Spark: Filter operation took 20.05 seconds


# long_trips_count=long_trips.count() # why do I need this?

Spark transformations like filter() are lazy. They only build a query plan and don’t touch any data. count() is an action, so it forces Spark to read the file, decompress it, and apply the filter. Without it, your “Spark: Filter operation took X seconds” would measure only the time to build the plan, which is a few milliseconds, and it would make Spark look absurdly fast. Pandas executes eagerly, so its timing is real work. Using count() (or .collect(), .show(), .write) is what makes the comparison fair.

In [8]:
# To calculate average trip distance
start_time=time.time()
avg_distance=df.select(mean('trip_distance')).collect()[0][0] # what does this line do? what is [0][0] after collect?
agg_time=time.time()-start_time
print(f"Spark: Aggregation took {agg_time:.2f} seconds")
print(f"Spark: Average trip distance is {avg_distance:.2f} miles")

Spark: Aggregation took 20.45 seconds
Spark: Average trip distance is 2.80 miles


### What does `df.select(mean('trip_distance')).collect()[0][0]` do?

Reading the line left to right:

1. **`df.select(mean('trip_distance'))`** is a lazy transformation. It builds a plan for a new DataFrame with one row and one column holding the average of `trip_distance`. Nothing runs yet.
2. **`.collect()`** is the action. It runs the job and returns the results to the driver as a Python **list of `Row` objects**. An aggregate with no `groupBy` returns exactly one row, so the list has one element.
3. **`[0]`** takes the first `Row` in that list.
4. **The second `[0]`** takes the first column of that `Row`, which is the actual Python float.

In [9]:
# Pandas: Data loaded in 16.10 seconds
# Pandas: Filter operation took 0.15 seconds
# Pandas: Aggregation took 0.03 seconds
# Pandas: Average trip distance is 2.80 miles

# Spark: Data loaded in 19.64 seconds
# Spark: Filter operation took 8.18 seconds
# Spark: Aggregation took 9.14 seconds
# Spark: Average trip distance is 2.80 miles

# Dataset1 > 1GB
# Dataset2 < 1GB

In [10]:
# To compare memory usage
%pip install psutil

In [11]:
# restart the kernel before you run this cell

# To monitor memory usage in Pandas
import pandas as pd
import psutil
import time
import os
import gc

# Function to get memory usage in MB
def get_memory_usage():
    gc.collect()
    process=psutil.Process(os.getpid())

    return process.memory_info().rss/1024/1024 # To convert to MB

# To track memory before loading
mem_before=get_memory_usage()

# To load dataset
start_time=time.time()
df=pd.read_csv("yellow_tripdata_2019-01.csv.gz",compression='gzip')
load_time=time.time()-start_time
mem_after_load=get_memory_usage()

# To filter trips longer than 10 miles
start_time=time.time()
long_trips=df[df['trip_distance']>10]
filter_time=time.time()-start_time
mem_after_filter=get_memory_usage()

# To calculate average trip distance
start_time=time.time()
avg_distance=df['trip_distance'].mean()
agg_time=time.time()-start_time
mem_after_agg=get_memory_usage()

# To print results
print(f"Pandas: Data loaded in {load_time:.2f} seconds, Memory: {mem_after_load-mem_before:.2f} MB")
print(f"Pandas: Filter operation took {filter_time:.2f} seconds, Memory: {mem_after_filter-mem_after_load:.2f} MB")
print(f"Pandas: Aggregation took {agg_time:.2f} seconds, Memory: {mem_after_agg-mem_after_filter:.2f} MB")

Pandas: Data loaded in 30.29 seconds, Memory: 1438.48 MB
Pandas: Filter operation took 0.16 seconds, Memory: 0.00 MB
Pandas: Aggregation took 0.02 seconds, Memory: 0.00 MB


In [12]:
# To monitor memory usage in Spark

# To monitor memory usage in Spark
# (restart the kernel before you run this cell, and do not run the pandas memory cell first)

import gc
import os
import time
import psutil
from pyspark.sql import SparkSession
from pyspark.sql.functions import mean

# Function to get total memory usage in MB (Python process + the Spark JVM child process)
def get_total_memory_usage():
    gc.collect()
    process=psutil.Process(os.getpid())
    processes=[process]+process.children(recursive=True)
    return sum(p.memory_info().rss for p in processes)/1024/1024 # To convert to MB

# To initialize SparkSession (this starts the JVM, so do it before taking the baseline)
spark=SparkSession.builder.appName("NYC_Taxi_Memory_Comparison").getOrCreate()
# Optional: give the JVM more room so the cached data fits in memory
# spark=SparkSession.builder.appName("NYC_Taxi_Memory_Comparison").config("spark.driver.memory","4g").getOrCreate()

# To track memory before loading
mem_before=get_total_memory_usage()

# To load dataset (cache + count forces Spark to read the data into memory)
start_time=time.time()
sdf=spark.read.csv("yellow_tripdata_2019-01.csv.gz",header=True,inferSchema=True)
sdf.cache()
sdf.count()
load_time=time.time()-start_time
mem_after_load=get_total_memory_usage()

# To filter trips longer than 10 miles
start_time=time.time()
long_trips_count=sdf.filter(sdf['trip_distance']>10).count()
filter_time=time.time()-start_time
mem_after_filter=get_total_memory_usage()

# To calculate average trip distance
start_time=time.time()
avg_distance=sdf.select(mean('trip_distance')).collect()[0][0]
agg_time=time.time()-start_time
mem_after_agg=get_total_memory_usage()

# To print results
print(f"Spark: Data loaded in {load_time:.2f} seconds, Memory: {mem_after_load-mem_before:.2f} MB")
print(f"Spark: Filter operation took {filter_time:.2f} seconds, Memory: {mem_after_filter-mem_after_load:.2f} MB")
print(f"Spark: Aggregation took {agg_time:.2f} seconds, Memory: {mem_after_agg-mem_after_filter:.2f} MB")
print(f"Spark: Average trip distance is {avg_distance:.2f} miles")

Spark: Data loaded in 132.41 seconds, Memory: 320.96 MB
Spark: Filter operation took 2.87 seconds, Memory: 92.72 MB
Spark: Aggregation took 2.01 seconds, Memory: -50.02 MB
Spark: Average trip distance is 2.80 miles
